# NB00 — WaTEM/SEDEM on Colab: Langegracht
Reference annual erosion/sediment model and grass-strip scenario, followed by an illustrative C-factor reduction. Adapted from the official MIT-licensed `original/api.ipynb`. Inputs and outputs persist on Drive; computation is local. Run top to bottom on a CPU runtime. No CN runoff in NB00. Installation and execution require validation on Colab.

## NB00.01 — Mount Drive and define paths

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, shutil, subprocess, json, hashlib, time, platform
from datetime import datetime, timezone
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Watem')
DATA = ROOT / 'data/tutorial/langegracht'
OUTPUT = ROOT / 'outputs/tutorial'
SOFTWARE = ROOT / 'software'
for p in (DATA.parent, OUTPUT, SOFTWARE): p.mkdir(parents=True, exist_ok=True)
REV = '226a7c06ae836f9736c1a88b74b5346033a2a8d6'
REPO = Path('/content/pywatemsedem')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content/watem_runs') / RUN_ID
WORK.mkdir(parents=True)
def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str,cmd)), check=True, **kwargs)
print(ROOT, sys.version, platform.machine())

## NB00.02 — Install software and pin source
The upstream changelog specifies WaTEM/SEDEM 5.0.3 and SAGA-WaTEM 2025.6.11. Use the matching Ubuntu package, not stock SAGA. Python dependencies are resolved against the Colab interpreter rather than forcing the upstream Python 3.13 lockfile. Save actual versions in the output manifest.

In [ ]:
import urllib.request
os_release = dict(line.strip().split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = os_release.get('VERSION_CODENAME','').strip('"')
assert platform.machine() == 'x86_64', 'Expected x86_64 Colab runtime'
assert codename in {'jammy','noble','focal','bookworm'}, f'Unsupported OS for pinned SAGA release: {codename}'
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','git-lfs','saga'])
run(['git','lfs','install'])
def download(url, path):
    if not path.exists():
        tmp = path.with_suffix(path.suffix+'.part')
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(path)
    return path
saga_name = f'saga-vpo_2025.6.11-1-ge95f136.{codename}_amd64.deb'
saga_deb = download('https://github.com/DOV-Vlaanderen/saga-watem/releases/download/2025.6.11/'+saga_name, SOFTWARE/saga_name)
local_deb = Path('/tmp') / saga_name
shutil.copy2(saga_deb, local_deb)
run(['apt-get','install','-y',str(local_deb)])
ws = download('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem', SOFTWARE/'watem_sedem_5.0.3')
binary = Path('/usr/local/bin/watem_sedem')
shutil.copy2(ws,binary); binary.chmod(0o755)
os.environ['WATEMSEDEM'] = str(binary.parent)
# saga-vpo contains plugins, not the SAGA command-line program.
# Ubuntu's saga package supplies saga_cmd.
saga_cmd = shutil.which('saga_cmd')
if not saga_cmd:
    run(['dpkg','-L','saga'])
    raise RuntimeError('saga_cmd missing after installing saga')
assert shutil.which('watem_sedem'), 'watem_sedem missing from PATH'
print('SAGA executable:', saga_cmd)
print('WaTEM executable:', shutil.which('watem_sedem'))
plugin_files = subprocess.check_output(['dpkg','-L','saga-vpo'],text=True).splitlines()
print('Installed SAGA plugin libraries:', [p for p in plugin_files if p.endswith('.so')])
run(['saga_cmd','--version'])
for args in (['saga_cmd','topology'], ['saga_cmd','watem']):
    p = subprocess.run(args,capture_output=True,text=True)
    msg = p.stdout+p.stderr
    print(msg[:3500])
    assert 'error: select a library' not in msg.lower() and 'cannot load' not in msg.lower(), f'Missing/incompatible SAGA library: {args}'
run(['ldd',str(binary)])
if not REPO.exists():
    run(['git','clone','https://github.com/watem-sedem/pywatemsedem.git',REPO],
        env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run(['git','-C',REPO,'checkout',REV],env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run([sys.executable,'-m','pip','install',str(REPO)])
import pywatemsedem
print('pywatemsedem',pywatemsedem.__version__)

## NB00.03 — Retrieve and verify official tutorial data
Cache only the required tutorial inputs and their shapefile sidecars. Validate every Git LFS file against its SHA256 pointer; a missing or incomplete cache is rebuilt without silently using old inputs.

In [ ]:
required = ['dtm.tif','K3.tif','basemap_landuse.tif','userchoices.ini']
stems = ['catchm_langegracht','river','infrastructure','roads','water','parcels','grass_strips','buffers','force_routing','outlets']
source = REPO/'tests/data'
names = sorted(p.name for p in source.iterdir() if p.is_file() and (p.name in required or p.stem in stems))
assert set(required).issubset(names)
for stem in stems:
    assert all(stem+ext in names for ext in ('.shp','.shx','.dbf','.prj')), stem
def expected_hash(name):
    raw = subprocess.check_output(['git','-C',str(REPO),'show',f'{REV}:tests/data/{name}'])
    if raw.startswith(b'version https://git-lfs'):
        return next(x.split(':',1)[1] for x in raw.decode().splitlines() if x.startswith('oid sha256:'))
    return hashlib.sha256(raw).hexdigest()
hashes = {name:expected_hash(name) for name in names}
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

def valid_file(path, name):
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == hashes[name]

def valid(folder):
    return all(valid_file(folder/n,n) for n in names)

DATA.mkdir(parents=True,exist_ok=True)
missing = [n for n in names if not valid_file(DATA/n,n)]
print(f'Valid cached files: {len(names)-len(missing)}/{len(names)}')
if missing:
    # Git LFS pull can fail at a detached HEAD or due to upstream LFS service.
    # Capture stderr in the notebook, rather than only showing CalledProcessError.
    pending = [n for n in missing if not valid_file(source/n,n)]
    if pending:
        p = subprocess.run(['git','-C',str(REPO),'lfs','pull','origin',
                            '--include='+','.join('tests/data/'+n for n in pending)],
                           capture_output=True,text=True)
        print('Git LFS return code:',p.returncode)
        print(p.stdout[-4000:])
        print(p.stderr[-6000:])
    def retrieve(name):
        if valid_file(DATA/name,name):
            return name,'cached'
        if valid_file(source/name,name):
            shutil.copy2(source/name,DATA/name)
            return name,'git-lfs'
        # GitHub media endpoint serves actual LFS objects rather than pointers.
        # Always require the exact SHA256 from the pinned upstream Git tree.
        errors=[]
        tmp=DATA/(name+'.part')
        for base in (
            f'https://media.githubusercontent.com/media/watem-sedem/pywatemsedem/{REV}/tests/data/',
            f'https://github.com/watem-sedem/pywatemsedem/raw/{REV}/tests/data/',
            f'https://raw.githubusercontent.com/watem-sedem/pywatemsedem/{REV}/tests/data/',
        ):
            for attempt in range(2):
                try:
                    req=urllib.request.Request(base+name,headers={'User-Agent':'Watem-Colab/1.0'})
                    with urllib.request.urlopen(req,timeout=60) as response, tmp.open('wb') as dst:
                        shutil.copyfileobj(response,dst)
                    if not valid_file(tmp,name):
                        raise ValueError('SHA256 mismatch (possibly a Git LFS pointer)')
                    tmp.replace(DATA/name)
                    return name,'direct download'
                except Exception as e:
                    errors.append(f'{base}: {type(e).__name__}: {e}')
                    tmp.unlink(missing_ok=True)
                    if attempt==0: time.sleep(2)
        raise RuntimeError(name+': '+' | '.join(errors))
    failures=[]
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures={pool.submit(retrieve,name):name for name in missing}
        for f in as_completed(futures):
            try:
                print(*f.result())
            except Exception as e:
                failures.append(str(e))
                print('FAILED:',e)
    if failures:
        raise RuntimeError(f'{len(failures)} data downloads failed. Valid files are retained on Drive. See HTTP/LFS diagnostics above.')
assert valid(DATA), 'Tutorial data checksum validation failed'
(DATA/'source_manifest.json').write_text(json.dumps({'revision':REV,'sha256':hashes},indent=2))
inputdata_folder = WORK/'inputs'
inputdata_folder.mkdir(exist_ok=True)
for n in names: shutil.copy2(DATA/n,inputdata_folder/n)
print(f'Validated {len(names)} files in {DATA}')
print('Local working data:', inputdata_folder)

## NB00.04 — Inspect input maps

In [ ]:
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from rasterio.plot import show
fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,name in zip(axes,['dtm.tif','K3.tif','basemap_landuse.tif']):
    with rasterio.open(inputdata_folder/name) as ds:
        print(name,ds.crs,ds.shape,ds.nodata)
        show(ds,ax=ax)
    ax.set_title(name)
plt.tight_layout(); plt.show()
parcels=gpd.read_file(inputdata_folder/'parcels.shp')
print(parcels.columns.tolist(),len(parcels))
parcels.plot(column='LANDUSE',legend=True,figsize=(7,7)); plt.show()

## NB00.05 — Initialize catchment (official tutorial parameters)

In [ ]:
from pywatemsedem.catchment import Catchment
catchment = Catchment('langegracht',inputdata_folder/'catchm_langegracht.shp',
    inputdata_folder/'dtm.tif',20,31370,-9999,results_folder=WORK)
catchment.kfactor=inputdata_folder/'K3.tif'
catchment.landuse=inputdata_folder/'basemap_landuse.tif'
catchment.vct_river=inputdata_folder/'river.shp'
catchment.vct_infrastructure_buildings=inputdata_folder/'infrastructure.shp'
catchment.vct_infrastructure_roads=inputdata_folder/'roads.shp'
catchment.vct_water=inputdata_folder/'water.shp'
catchment.dtm.plot()
catchment.segments.plot(nodata=0)

### NB00.05B — Scientific input atlas and inventory

In [ ]:
import pandas as pd
from IPython.display import display
from matplotlib.colors import ListedColormap, BoundaryNorm, TwoSlopeNorm
from matplotlib.patches import Patch
from rasterio.transform import array_bounds
FIG=WORK/'figures'; TAB=WORK/'tables'
for folder in (FIG,TAB): folder.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':110,'axes.titlesize':11,'font.size':10})
def finish(fig,name):
    fig.savefig(FIG/(name+'.png'),dpi=220,bbox_inches='tight')
    plt.show()
    plt.close(fig)
def table(df,name):
    display(df)
    df.to_csv(TAB/(name+'.csv'),index=False)
def arr(obj):
    a=np.asarray(obj.arr,dtype=float)
    return np.ma.masked_where(~np.isfinite(a)|(a==obj.rp.nodata),a)
mask=np.asarray(catchment.mask.arr)==1
def masked(a):
    a=np.ma.asarray(a,dtype=float)
    return np.ma.masked_where(~mask|~np.isfinite(a.data)|np.ma.getmaskarray(a),a)
with rasterio.open(inputdata_folder/'dtm.tif') as ds:
    print('Source DEM:',ds.crs,ds.res)
boundary=gpd.read_file(inputdata_folder/'catchm_langegracht.shp').to_crs(31370)
parcels=gpd.read_file(inputdata_folder/'parcels.shp').to_crs(31370)
river=gpd.read_file(inputdata_folder/'river.shp').to_crs(31370)
roads=gpd.read_file(inputdata_folder/'roads.shp').to_crs(31370)
buildings=gpd.read_file(inputdata_folder/'infrastructure.shp').to_crs(31370)
water=gpd.read_file(inputdata_folder/'water.shp').to_crs(31370)
strips=gpd.read_file(inputdata_folder/'grass_strips.shp').to_crs(31370)
# Use grid coordinates for raster panels, projected metres for vector panels.
def raster_panel(fig,ax,a,title,cmap='viridis',vmin=None,vmax=None,unit=''):
    im=ax.imshow(masked(a),cmap=cmap,vmin=vmin,vmax=vmax,interpolation='nearest')
    ax.set_title(title); ax.set_axis_off()
    fig.colorbar(im,ax=ax,shrink=.73,label=unit)
    return im
fig,ax=plt.subplots(2,3,figsize=(15,10),constrained_layout=True)
raster_panel(fig,ax[0,0],arr(catchment.dtm),'A. Elevation','terrain',unit='m')
raster_panel(fig,ax[0,1],arr(catchment.kfactor),'B. Soil erodibility K','YlOrBr',unit='Native K units')
raster_panel(fig,ax[0,2],arr(catchment.landuse),'C. Background land-use codes','tab20',unit='Class code')
boundary.boundary.plot(ax=ax[1,0],color='black',linewidth=1)
parcels.plot(ax=ax[1,0],column='C_crop',cmap='YlOrBr',vmin=0,vmax=1,legend=True,edgecolor='grey',linewidth=.2)
ax[1,0].set_title('D. Parcel crop C-factor')
boundary.boundary.plot(ax=ax[1,1],color='black')
river.plot(ax=ax[1,1],color='#247ba0',linewidth=1.3)
roads.plot(ax=ax[1,1],color='#777777',linewidth=.8)
buildings.plot(ax=ax[1,1],color='#a95b43')
water.plot(ax=ax[1,1],color='#5fb8cf')
ax[1,1].legend(handles=[Patch(color=x,label=y) for x,y in [('#247ba0','River'),('#777777','Road'),('#a95b43','Building'),('#5fb8cf','Water')]],loc='best',fontsize=8)
ax[1,1].set_title('E. Drainage and infrastructure')
boundary.boundary.plot(ax=ax[1,2],color='black')
parcels.boundary.plot(ax=ax[1,2],color='grey',linewidth=.3)
strips.plot(ax=ax[1,2],color='#37834b')
ax[1,2].set_title('F. Proposed grass strips (S2)')
for a in ax[1]: a.set_axis_off(); a.set_aspect('equal')
fig.suptitle('Langegracht | spatial model inputs',fontsize=16)
finish(fig,'01_spatial_inputs')
area=mask.sum()*20*20/10000
table(pd.DataFrame([{'resolution_m':20,'grid_cells_in_mask':int(mask.sum()),'model_area_ha':area,'vector_area_ha':boundary.geometry.area.sum()/10000,'parcels':len(parcels),'grass_strip_features':len(strips)}]),'01_catchment_summary')
p=parcels.copy(); p['area_ha']=p.geometry.intersection(boundary.geometry.union_all()).area/10000
table(p.groupby('LANDUSE',dropna=False).agg(parcels=('area_ha','size'),area_ha=('area_ha','sum'),mean_C_crop=('C_crop','mean')).reset_index(),'02_parcel_inventory')
table(pd.DataFrame([{'input':label,'min':float(a.min()),'median':float(np.ma.median(a)),'max':float(a.max())} for label,obj in [('Elevation',catchment.dtm),('K factor',catchment.kfactor)] for a in [masked(arr(obj))]]),'03_input_statistics')

## NB00.06 — Read official model settings
No calibration or hydrological assumptions are added.

In [ ]:
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.scenario import Scenario
default_choices=inputdata_folder/'userchoices.ini'
options=Options(); options.read_values_from_ini(default_choices)
parameters=Parameters(); parameters.read_values_from_ini(default_choices)
extensions=Extensions(); extensions.read_values_from_ini(default_choices)
output=Output(); output.read_values_from_ini(default_choices)
extensionparameters=ExtensionsParameters(extensions)
extensionparameters.apply_defaults()
for p in (extensionparameters.ktc_low,extensionparameters.ktc_high,extensionparameters.ktc_limit):
    p.read_value_from_ini(default_choices)
choices=Choices(options,parameters,extensions,extensionparameters,output)
assert not choices.extensions.curve_number.value, 'NB00 expects standard erosion model'
print(default_choices.read_text())

### NB00.06B — Effective parameter table

In [ ]:
from pywatemsedem.choices import UserChoice
parameter_rows=[]
for section,container in [('Options',options),('Parameters',parameters),('Extensions',extensions),('Extension parameters',extensionparameters),('Output',output)]:
    for v in vars(container).values():
        if isinstance(v,UserChoice):
            parameter_rows.append({'section':section,'parameter':v.key,'value':v.value})
table(pd.DataFrame(parameter_rows),'04_effective_model_settings')
print('Original configuration:'); print(default_choices.read_text())

## NB00.07 — Prepare baseline and execute
The upstream wrapper's run_model does not check subprocess return codes. We use the same executable and generated INI, but explicitly check return status and output existence.

In [ ]:
def prepare(scenario, source_measures=False):
    scenario.composite_landuse=scenario.create_composite_landuse()
    scenario.cfactor=scenario.create_cfactor(use_source_oriented_measures=source_measures)
    ep=scenario.choices.extensionparameters
    scenario.ktc=scenario.create_ktc(ep.ktc_low.value,ep.ktc_high.value,ep.ktc_limit.value,
        not scenario.choices.extensions.create_ktc_map.value)
    scenario.prepare_input_files()
    scenario.create_ini_file()
    return scenario
def execute(scenario):
    dest=scenario.sfolder.wsoutput_folder
    assert not any(dest.iterdir()), f'Output directory already populated: {dest}; start a fresh run'
    p=subprocess.run([str(binary),str(scenario.ini)],capture_output=True,text=True,errors='replace')
    log=scenario.sfolder.scenario_folder/'model.log'
    log.write_text(p.stdout+'\n'+p.stderr)
    print(p.stdout[-5000:],p.stderr[-3000:])
    p.check_returncode()
    rasters=list(dest.glob('*.rst'))
    assert rasters, f'No output rasters: see {log}'
    print('Outputs:',sorted(x.name for x in dest.iterdir()))
    return scenario
baseline=Scenario(catchment,2019,1,choices)
baseline.vct_parcels=inputdata_folder/'parcels.shp'
prepare(baseline)
baseline.cfactor.plot()
execute(baseline)

## NB00.08 — Management scenarios
S2 reproduces the upstream grass-strip example. S4 uses the upstream illustrative 80% C-factor reduction on agricultural parcels, not an empirically established regenerative-agriculture effect. The original tutorial comments out S4 execution; here we explicitly attempt it.

In [ ]:
grass=Scenario(catchment,2019,2,choices)
grass.vct_parcels=inputdata_folder/'parcels.shp'
grass.vct_grass_strips=inputdata_folder/'grass_strips.shp'
execute(prepare(grass))
reduced=Scenario(catchment,2019,4,choices)
modified=gpd.read_file(inputdata_folder/'parcels.shp')
modified.loc[modified['LANDUSE']==-9999,'C_reduct']=0.8
reduced.vct_parcels=modified
execute(prepare(reduced,source_measures=True))
scenarios={'baseline':baseline,'grass_strips':grass,'C_reduction':reduced}

### NB00.08B — Scenario setup and intervention QA

In [ ]:
# All comparisons use the same catchment grid and mask.
labels={'baseline':'S1 | Baseline','grass_strips':'S2 | Grass strips','C_reduction':'S4 | C reduction'}
C={k:masked(arr(s.cfactor)) for k,s in scenarios.items()}
L={k:masked(arr(s.composite_landuse)) for k,s in scenarios.items()}
KTC={}
for k,s in scenarios.items():
    if not s.ktc.is_empty(): KTC[k]=masked(arr(s.ktc))
def delta(a,b):
    return np.ma.asarray(a)-np.ma.asarray(b)
def abs_lim(arrays,pct=100):
    vals=np.concatenate([np.abs(np.ma.asarray(a).compressed()) for a in arrays])
    return max(float(np.percentile(vals,pct)),1e-9) if vals.size else 1
def shared_row(fig,axes,arrays,titles,cmap,vmin,vmax,unit):
    for ax,a,title in zip(axes,arrays,titles):
        im=ax.imshow(masked(a),cmap=cmap,vmin=vmin,vmax=vmax,interpolation='nearest')
        ax.set_title(title); ax.set_axis_off()
    fig.colorbar(im,ax=list(axes),shrink=.65,label=unit)
titles=[labels[k] for k in scenarios]
fig,axes=plt.subplots(3,3,figsize=(15,13),constrained_layout=True)
shared_row(fig,axes[0],list(C.values()),titles,'YlOrBr',0,1,'C-factor')
dC=[delta(a,C['baseline']) for a in C.values()]
lim=abs_lim(dC)
shared_row(fig,axes[1],dC,['C change: '+t for t in titles],'RdBu_r',-lim,lim,'Scenario minus baseline')
shared_row(fig,axes[2],[np.ma.array(a==-6,mask=np.ma.getmaskarray(a)) for a in L.values()],['Grass cells: '+t for t in titles],'Greens',0,1,'0 = no; 1 = grass strip')
fig.suptitle('Management interventions | model-ready input maps',fontsize=16)
finish(fig,'02_scenario_setup')
if len(KTC)==len(scenarios):
    fig,axes=plt.subplots(2,3,figsize=(15,8),constrained_layout=True)
    # Routing sentinel 9999 is not a physical transport coefficient.
    kt={k:np.ma.masked_where(a==9999,a) for k,a in KTC.items()}
    vmax=max(float(a.max()) for a in kt.values())
    shared_row(fig,axes[0],list(kt.values()),titles,'viridis',0,max(vmax,1e-9),'kTC (routing sentinel masked)')
    d=[delta(a,kt['baseline']) for a in kt.values()]; lim=abs_lim(d)
    shared_row(fig,axes[1],d,['kTC change: '+t for t in titles],'RdBu_r',-lim,lim,'Scenario minus baseline')
    finish(fig,'03_transport_coefficient')
else:
    print('kTC map generated internally by model; see Capacity output instead.')
rows=[]
for k in scenarios:
    a=C[k]; d=delta(a,C['baseline'])
    valid=~np.ma.getmaskarray(d)
    row={'scenario':labels[k],'mean_C':float(a.mean()),'grass_cells':int((L[k]==-6).sum()),'C_changed_cells':int(np.count_nonzero(valid & (np.abs(d.data)>1e-7))),'C_decreased_cells':int(np.count_nonzero(valid & (d.data<-1e-7)))}
    if k in KTC and 'baseline' in KTC:
        dk=delta(KTC[k],KTC['baseline']); row['kTC_changed_cells']=int(np.count_nonzero(~np.ma.getmaskarray(dk)&(np.abs(dk.data)>1e-7)))
    rows.append(row)
qa=pd.DataFrame(rows)
table(qa,'05_scenario_input_checks')
for k,s in scenarios.items():
    a=L[k].compressed()
    print(k,'composite raster range:',a.min(),a.max(),'fits signed Int16:',bool(np.all((a>=-32768)&(a<=32767))))
if qa.loc[qa.scenario==labels['grass_strips'],'grass_cells'].iloc[0]==0:
    print('WARNING: S2 has no grass-strip cells; inspect rasterization before interpreting results.')
if qa.loc[qa.scenario==labels['C_reduction'],'C_decreased_cells'].iloc[0]==0:
    print('WARNING: S4 did not decrease any C-factor cells.')

## NB00.09 — Comparable output maps and raster diagnostics

In [ ]:
# Model output definitions: official WaTEM/SEDEM 5.0.3 docs/output.rst.
# Negative WATEREROS = erosion; positive = deposition.
def load_output(s,name):
    p=s.sfolder.wsoutput_folder/name
    with rasterio.open(p) as ds:
        a=ds.read(1,masked=True)
        assert a.shape==mask.shape,(name,a.shape,mask.shape)
        return masked(a)
def export_output(s,name,target):
    p=s.sfolder.wsoutput_folder/name
    with rasterio.open(p) as ds:
        profile=ds.profile.copy(); profile.update(driver='GTiff',compress='deflate')
        with rasterio.open(s.sfolder.scenario_folder/target,'w',**profile) as dst: dst.write(ds.read(1),1)
M={}
for k,s in scenarios.items():
    M[k]={}
    for key,name in [('net_kg','WATEREROS (kg per gridcel).rst'),('net_mm','WATEREROS (mm per gridcel).rst'),('rusle','RUSLE.rst'),('capacity','Capacity.rst'),('river','SediExport_kg.rst'),('cumulative','cumulative.rst')]:
        if (s.sfolder.wsoutput_folder/name).exists():
            M[k][key]=load_output(s,name)
            export_output(s,name,key+'.tif')
# 20 m cell: kg/cell * 10 / 400 = t/ha; annual model.
net={k:m['net_kg']*10/400 for k,m in M.items()}
fig,axes=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
lim=abs_lim(list(net.values()),98)
shared_row(fig,axes[0],list(net.values()),titles,'RdBu',-lim,lim,'Net redistribution (t/ha/yr); − erosion, + deposition')
d=[delta(a,net['baseline']) for a in net.values()]; lim=abs_lim(d,98)
shared_row(fig,axes[1],d,['Change: '+t for t in titles],'RdBu',-lim,lim,'Scenario − baseline (t/ha/yr)')
fig.suptitle('Net water erosion and deposition | shared scales (98th percentile clipping)',fontsize=15)
finish(fig,'04_net_redistribution')
for key,unit,mult,cmap in [('rusle','Potential soil loss (t/ha/yr)',10,'YlOrRd'),('capacity','Transport capacity (kg/cell/yr)',1,'magma'),('river','Sediment delivered to river (kg/yr per cell)',1,'Blues')]:
    if not all(key in m for m in M.values()): continue
    data=[m[key]*mult for m in M.values()]
    vmax=abs_lim(data,99)
    fig,axes=plt.subplots(1,3,figsize=(15,5),constrained_layout=True)
    shared_row(fig,axes,data,titles,cmap,0,vmax,unit)
    fig.suptitle(unit+' | shared scale, clipped at pooled 99th percentile')
    finish(fig,'05_'+key)
# Mass sums exclude river pixels, outside-domain cells, and use common valid land footprint.
land=mask & ~np.isin(L['baseline'].data,[-1,-2,-5])
common=land & np.logical_and.reduce([~np.ma.getmaskarray(m['net_kg']) for m in M.values()])
rows=[]
for k,m in M.items():
    x=m['net_kg'].data[common]
    rows.append({'scenario':labels[k],'common_land_ha':common.sum()*400/10000,'erosion_sum_t':-x[x<0].sum()/1000,'deposition_sum_t':x[x>0].sum()/1000,'signed_net_t':x.sum()/1000})
table(pd.DataFrame(rows),'06_common_land_raster_diagnostics')
print('Raster diagnostics are not the official catchment sediment budget.')

### NB00.09B — Official sediment budgets and management effect

In [ ]:
import re
# Parse the model's own budget, retain raw reports and never substitute raster sums.
patterns={'erosion':r'^total erosion\b','deposition':r'^total deposition\b','river_delivery':r'^sediment leaving the catchment\s*,?\s*via the river\b','other_export':r'^sediment leaving the catchment\s*,?\s*not via the river\b','buffer_trapped':r'^sediment trapped in buffers\b','sewer':r'^sediment entering sewer system\b'}
def parse_budget(text):
    result={}
    for line in text.splitlines():
        for key,pat in patterns.items():
            if re.search(pat,line.strip(),re.I):
                # The last numeric token is the value (supports decimal comma and exponent).
                values=re.findall(r'[-+]?\d+(?:[.,]\d+)?(?:[Ee][-+]?\d+)?',line)
                if values: result[key]=float(values[-1].replace(',','.'))
    return result
reports={}; rows=[]
for k,s in scenarios.items():
    p=s.sfolder.wsoutput_folder/'Total sediment.txt'
    if not p.exists(): print('Missing:',p); continue
    raw=p.read_text(errors='replace'); reports[k]=raw
    print('\n---',labels[k],'---\n',raw)
    b=parse_budget(raw)
    if not {'erosion','deposition','river_delivery','other_export'}.issubset(b):
        print('WARNING: Unrecognized budget format; raw report preserved. Parsed:',b)
        continue
    e=abs(b['erosion']); d=b['deposition']; riverkg=b['river_delivery']; other=b['other_export']
    residual=e-d-riverkg-other-b.get('buffer_trapped',0)-b.get('sewer',0)
    rows.append({'scenario':labels[k],'erosion_t_yr':e/1000,'deposition_t_yr':d/1000,'river_delivery_t_yr':riverkg/1000,'other_export_t_yr':other/1000,'buffer_trapped_t_yr':b.get('buffer_trapped',0)/1000,'sewer_t_yr':b.get('sewer',0)/1000,'balance_residual_t_yr':residual/1000,'river_delivery_ratio_pct':100*riverkg/e if e else np.nan})
(WORK/'sediment_budget_reports.json').write_text(json.dumps(reports,indent=2))
budget=pd.DataFrame(rows)
if len(budget):
    for col in ['erosion_t_yr','river_delivery_t_yr']:
        ref=budget.loc[budget.scenario==labels['baseline'],col]
        budget[col.replace('_t_yr','_reduction_pct')]=100*(1-budget[col]/ref.iloc[0]) if len(ref) and ref.iloc[0]!=0 else np.nan
    table(budget,'07_official_sediment_budget')
    fig,ax=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
    budget.set_index('scenario')[['erosion_t_yr','deposition_t_yr']].plot.bar(ax=ax[0],color=['#a65b3e','#71935c'],rot=0)
    budget.set_index('scenario')[['river_delivery_t_yr','other_export_t_yr']].plot.bar(ax=ax[1],color=['#387ba1','#999999'],rot=0)
    for a,title in zip(ax,['Landscape erosion and deposition','Sediment delivery and export']):
        a.set_title(title); a.set_ylabel('t/year'); a.set_xlabel(''); a.legend(fontsize=8); a.grid(axis='y',alpha=.2)
    finish(fig,'06_sediment_budget')
    fig,ax=plt.subplots(figsize=(9,4),constrained_layout=True)
    budget.set_index('scenario')[['erosion_reduction_pct','river_delivery_reduction_pct']].plot.bar(ax=ax,rot=0,color=['#a65b3e','#387ba1'])
    ax.axhline(0,color='black',lw=.8); ax.set_ylabel('Reduction relative to baseline (%)'); ax.set_xlabel('')
    ax.legend(['Erosion','River delivery']); ax.grid(axis='y',alpha=.2)
    finish(fig,'07_management_effect')
else:
    print('No budget rows parsed; use raw reports above. No estimated values substituted.')
print('River delivery = sediment entering river cells, not necessarily measured outlet yield.')

## NB00.10 — Persist results and provenance
Each run gets a separate timestamped folder; no prior Drive results are deleted. Model INI files retain local execution paths and should be regenerated for reruns.

In [ ]:
manifest={'run_id':RUN_ID,'upstream_revision':REV,'data_sha256':hashes,
          'watem_sedem':'5.0.3','saga_watem':'2025.6.11','os':os_release,
          'python':sys.version,'pywatemsedem':pywatemsedem.__version__}
(WORK/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
(WORK/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
dest=OUTPUT/RUN_ID
shutil.copytree(WORK,dest,ignore=shutil.ignore_patterns('inputs'))
print('Saved results:',dest)